In [2]:
"""
Intelligent Sudoku Solver
Course: CSE 3812 - Artificial Intelligence Laboratory
University: United International University (UIU)

Techniques:
1. CSP Modeling
2. AC-3 Constraint Propagation
3. Backtracking Search
4. MRV Heuristic
5. Ablation Study

Input:
    puzzles.txt
    Each line contains an 81-character Sudoku string.
    0 represents an unassigned cell.

Output:
    solutions.txt
    Each solved puzzle is written as an 81-character string.

The program also performs an ablation study on the first puzzle:
    - MRV Enabled
    - MRV Disabled
"""

import time
from collections import deque


class SudokuCSP:
    """
    Represents a Sudoku puzzle as a Constraint Satisfaction Problem.

    Variables:
        81 cells represented by (row, column).

    Domains:
        Possible values for every cell.

    Constraints:
        Neighboring cells must contain different values.
    """

    DIGITS = set(range(1, 10))

    def __init__(self, puzzle):
        """
        Initialize the Sudoku CSP.

        puzzle:
            81-character string containing digits 0-9.
        """

        if len(puzzle) != 81:
            raise ValueError(
                "Invalid puzzle: Sudoku puzzle must contain exactly 81 characters."
            )

        if not all(char.isdigit() for char in puzzle):
            raise ValueError(
                "Invalid puzzle: input must contain only digits."
            )

        self.puzzle = puzzle

        # Create all 81 variables.
        self.variables = [
            (row, col)
            for row in range(9)
            for col in range(9)
        ]

        # Store the original board.
        self.board = [
            [int(puzzle[row * 9 + col]) for col in range(9)]
            for row in range(9)
        ]

        # Create neighbor relationships.
        self.neighbors = self.create_neighbors()

        # Create domains.
        self.domains = self.create_domains()

        # Search statistics.
        self.node_expansions = 0
        self.backtracks = 0

    # ---------------------------------------------------------
    # CSP INITIALIZATION
    # ---------------------------------------------------------

    def create_neighbors(self):
        """
        Create neighbors for every Sudoku variable.

        Two cells are neighbors if they share:
        - the same row,
        - the same column, or
        - the same 3x3 subgrid.
        """

        neighbors = {}

        for variable in self.variables:
            row, col = variable

            neighbor_set = set()

            # Same row
            for c in range(9):
                if c != col:
                    neighbor_set.add((row, c))

            # Same column
            for r in range(9):
                if r != row:
                    neighbor_set.add((r, col))

            # Same 3x3 subgrid
            start_row = (row // 3) * 3
            start_col = (col // 3) * 3

            for r in range(start_row, start_row + 3):
                for c in range(start_col, start_col + 3):
                    if (r, c) != variable:
                        neighbor_set.add((r, c))

            neighbors[variable] = neighbor_set

        return neighbors

    def create_domains(self):
        """
        Create initial domains.

        Given cells:
            domain = {given value}

        Empty cells:
            domain = {1,2,3,4,5,6,7,8,9}
        """

        domains = {}

        for variable in self.variables:
            row, col = variable
            value = self.board[row][col]

            if value == 0:
                domains[variable] = set(self.DIGITS)
            else:
                domains[variable] = {value}

        return domains

    # ---------------------------------------------------------
    # CONSTRAINT CHECKING
    # ---------------------------------------------------------

    def checkIfUsed(self, row, col, value):
        """
        Check whether value is already used in the
        same row, column, or 3x3 subgrid.

        Returns:
            True  -> value is already used
            False -> value is not used
        """

        # Check row
        for c in range(9):
            if self.board[row][c] == value:
                return True

        # Check column
        for r in range(9):
            if self.board[r][col] == value:
                return True

        # Check 3x3 subgrid
        start_row = (row // 3) * 3
        start_col = (col // 3) * 3

        for r in range(start_row, start_row + 3):
            for c in range(start_col, start_col + 3):
                if self.board[r][c] == value:
                    return True

        return False

    def checkAxes(self, variable, value):
        """
        Check whether assigning value to variable
        violates Sudoku constraints.

        Returns:
            True  -> assignment is valid
            False -> assignment is invalid
        """

        row, col = variable

        # Check row
        for c in range(9):
            if c != col and self.board[row][c] == value:
                return False

        # Check column
        for r in range(9):
            if r != row and self.board[r][col] == value:
                return False

        # Check 3x3 box
        start_row = (row // 3) * 3
        start_col = (col // 3) * 3

        for r in range(start_row, start_row + 3):
            for c in range(start_col, start_col + 3):
                if (r, c) != variable:
                    if self.board[r][c] == value:
                        return False

        return True

    # ---------------------------------------------------------
    # AC-3 ALGORITHM
    # ---------------------------------------------------------

    def revise(self, xi, xj):
        """
        REVISE operation of AC-3.

        Remove a value x from Xi if there is no value y
        in Xj such that Xi != Xj can be satisfied.

        Returns:
            True  -> domain of Xi was revised
            False -> no change
        """

        revised = False

        values_to_remove = set()

        for x in self.domains[xi]:

            # Check whether x has at least one supporting
            # value y in the domain of Xj.
            supported = False

            for y in self.domains[xj]:

                # Sudoku constraint:
                # neighboring cells must have different values.
                if x != y:
                    supported = True
                    break

            # If no supporting value exists, remove x.
            if not supported:
                values_to_remove.add(x)

        if values_to_remove:
            self.domains[xi] -= values_to_remove
            revised = True

        return revised

    def AC3(self):
        """
        AC-3 algorithm.

        Initially all arcs are placed into the queue.

        If REVISE changes Xi's domain, all arcs
        (Xk, Xi), where Xk is a neighbor of Xi,
        are added to the queue.

        Returns:
            True  -> CSP is arc consistent
            False -> inconsistency detected
        """

        queue = deque()

        # Add all arcs to the initial queue.
        for xi in self.variables:
            for xj in self.neighbors[xi]:
                queue.append((xi, xj))

        # Process the queue.
        while queue:

            xi, xj = queue.popleft()

            if self.revise(xi, xj):

                # If Xi's domain becomes empty,
                # the CSP is inconsistent.
                if len(self.domains[xi]) == 0:
                    return False

                # Add affected neighboring arcs.
                for xk in self.neighbors[xi]:

                    if xk != xj:
                        queue.append((xk, xi))

        return True

    # ---------------------------------------------------------
    # BACKTRACKING + MRV
    # ---------------------------------------------------------

    def is_complete(self):
        """
        Check whether every variable has exactly one value.
        """

        for variable in self.variables:
            if len(self.domains[variable]) != 1:
                return False

        return True

    def select_unassigned_variable(self, use_mrv=True):
        """
        Select the next variable.

        If MRV is enabled:
            Select variable with smallest domain.

        If MRV is disabled:
            Select the first unassigned variable
            using sequential ordering.
        """

        unassigned = [
            variable
            for variable in self.variables
            if len(self.domains[variable]) > 1
        ]

        if not unassigned:
            return None

        if use_mrv:
            # Minimum Remaining Values
            return min(
                unassigned,
                key=lambda variable: len(self.domains[variable])
            )

        # Sequential variable selection
        return unassigned[0]

    def consistent_with_assigned_neighbors(self, variable, value):
        """
        Check whether value conflicts with any currently
        assigned neighboring variable.
        """

        for neighbor in self.neighbors[variable]:

            if len(self.domains[neighbor]) == 1:

                neighbor_value = next(iter(self.domains[neighbor]))

                if value == neighbor_value:
                    return False

        return True

    def backtrack(self, use_mrv=True):
        """
        Recursive Backtracking Search.

        use_mrv=True:
            MRV heuristic is used.

        use_mrv=False:
            Sequential variable selection is used.
        """

        # If all variables are assigned, solution is found.
        if self.is_complete():
            return True

        # Select next variable.
        variable = self.select_unassigned_variable(use_mrv)

        if variable is None:
            return True

        # Save original domain so that it can be restored.
        original_domain = self.domains[variable].copy()

        # Try each possible value.
        for value in sorted(original_domain):

            # Count a node expansion.
            self.node_expansions += 1

            # Check consistency with current assignments.
            if not self.consistent_with_assigned_neighbors(
                variable, value
            ):
                continue

            # Save all domains before making the assignment.
            saved_domains = {
                var: domain.copy()
                for var, domain in self.domains.items()
            }

            # Assign value to selected variable.
            self.domains[variable] = {value}

            # Apply AC-3 after assignment.
            if self.AC3():

                # Continue recursively.
                if self.backtrack(use_mrv):
                    return True

            # Restore previous state.
            self.domains = saved_domains

            # A branch failed.
            self.backtracks += 1

        return False

    # ---------------------------------------------------------
    # SOLVE
    # ---------------------------------------------------------

    def solve(self, use_mrv=True):
        """
        Solve the Sudoku puzzle.

        Steps:
            1. AC-3 preprocessing
            2. Backtracking Search
            3. MRV if enabled
        """

        # Reset statistics.
        self.node_expansions = 0
        self.backtracks = 0

        # Initial AC-3 preprocessing.
        if not self.AC3():
            return None

        # Backtracking Search.
        if not self.backtrack(use_mrv):
            return None

        return self.get_solution()

    def get_solution(self):
        """
        Convert the final domains into an 81-character
        Sudoku solution string.
        """

        solution = ""

        for row in range(9):
            for col in range(9):

                variable = (row, col)

                if len(self.domains[variable]) != 1:
                    return None

                value = next(iter(self.domains[variable]))

                solution += str(value)

        return solution


# =============================================================
# FILE HANDLING
# =============================================================

def read_puzzles(filename):
    """
    Read all Sudoku puzzles from the input file.

    Blank lines are ignored.
    """

    puzzles = []

    with open(filename, "r") as file:

        for line in file:
            puzzle = line.strip()

            if puzzle:
                puzzles.append(puzzle)

    return puzzles


def write_solutions(filename, solutions):
    """
    Write solved Sudoku strings to solutions.txt.
    """

    with open(filename, "w") as file:

        for solution in solutions:
            file.write(solution + "\n")


# =============================================================
# ABLATION STUDY
# =============================================================

def run_ablation(puzzle):
    """
    Compare:
        1. MRV enabled
        2. MRV disabled

    Metrics:
        - Execution time
        - Node expansions
        - Backtracks
    """

    print("\n" + "=" * 60)
    print("ABLATION STUDY")
    print("=" * 60)

    # ---------------------------------------------------------
    # MRV ENABLED
    # ---------------------------------------------------------

    solver_mrv = SudokuCSP(puzzle)

    start_time = time.perf_counter()

    solution_mrv = solver_mrv.solve(use_mrv=True)

    end_time = time.perf_counter()

    mrv_time = end_time - start_time

    # ---------------------------------------------------------
    # MRV DISABLED
    # ---------------------------------------------------------

    solver_no_mrv = SudokuCSP(puzzle)

    start_time = time.perf_counter()

    solution_no_mrv = solver_no_mrv.solve(use_mrv=False)

    end_time = time.perf_counter()

    no_mrv_time = end_time - start_time

    # ---------------------------------------------------------
    # RESULTS
    # ---------------------------------------------------------

    print("\nConfiguration 1: MRV Enabled")
    print("-" * 40)

    print(f"Execution Time : {mrv_time:.6f} seconds")
    print(f"Node Expansions: {solver_mrv.node_expansions}")
    print(f"Backtracks     : {solver_mrv.backtracks}")
    print(f"Solved         : {solution_mrv is not None}")

    print("\nConfiguration 2: MRV Disabled")
    print("-" * 40)

    print(f"Execution Time : {no_mrv_time:.6f} seconds")
    print(f"Node Expansions: {solver_no_mrv.node_expansions}")
    print(f"Backtracks     : {solver_no_mrv.backtracks}")
    print(f"Solved         : {solution_no_mrv is not None}")

    # ---------------------------------------------------------
    # MATHEMATICAL COMPARISON
    # ---------------------------------------------------------

    print("\n" + "=" * 60)
    print("MATHEMATICAL COMPARISON")
    print("=" * 60)

    # Time improvement
    if no_mrv_time > 0:

        time_improvement = (
            (no_mrv_time - mrv_time)
            / no_mrv_time
        ) * 100

        print(
            f"\nExecution Time Improvement: "
            f"{time_improvement:.2f}%"
        )

    # Node reduction
    if solver_no_mrv.node_expansions > 0:

        node_reduction = (
            (
                solver_no_mrv.node_expansions
                - solver_mrv.node_expansions
            )
            / solver_no_mrv.node_expansions
        ) * 100

        print(
            f"Node Expansion Reduction: "
            f"{node_reduction:.2f}%"
        )

    # Backtrack reduction
    if solver_no_mrv.backtracks > 0:

        backtrack_reduction = (
            (
                solver_no_mrv.backtracks
                - solver_mrv.backtracks
            )
            / solver_no_mrv.backtracks
        ) * 100

        print(
            f"Backtrack Reduction: "
            f"{backtrack_reduction:.2f}%"
        )

    print("\n" + "=" * 60)


# =============================================================
# MAIN PROGRAM
# =============================================================

def main():

    input_file = "puzzles.txt"
    output_file = "solutions.txt"

    print("=" * 60)
    print("INTELLIGENT SUDOKU SOLVER")
    print("CSP + AC-3 + Backtracking + MRV")
    print("=" * 60)

    # Read puzzles.
    try:
        puzzles = read_puzzles(input_file)

    except FileNotFoundError:
        print(f"\nError: {input_file} was not found.")
        return

    if not puzzles:
        print("\nNo puzzles found in the input file.")
        return

    print(f"\nTotal puzzles found: {len(puzzles)}")

    solutions = []

    # ---------------------------------------------------------
    # SOLVE ALL PUZZLES
    # ---------------------------------------------------------

    for index, puzzle in enumerate(puzzles, start=1):

        print(f"\nSolving Puzzle {index}...")

        try:

            solver = SudokuCSP(puzzle)

            start_time = time.perf_counter()

            solution = solver.solve(use_mrv=True)

            end_time = time.perf_counter()

            execution_time = end_time - start_time

            if solution is None:

                print("Result: No solution found.")

                solutions.append("NO SOLUTION")

            else:

                print("Result: Solved")
                print(
                    f"Execution Time: "
                    f"{execution_time:.6f} seconds"
                )

                print(
                    f"Node Expansions: "
                    f"{solver.node_expansions}"
                )

                print(
                    f"Backtracks: "
                    f"{solver.backtracks}"
                )

                solutions.append(solution)

        except ValueError as error:

            print(f"Error in Puzzle {index}: {error}")

            solutions.append("INVALID PUZZLE")

    # ---------------------------------------------------------
    # WRITE OUTPUT
    # ---------------------------------------------------------

    write_solutions(output_file, solutions)

    print("\n" + "=" * 60)
    print(f"Solutions written to: {output_file}")
    print("=" * 60)

    # ---------------------------------------------------------
    # ABLATION STUDY
    # ---------------------------------------------------------
    #
    # The assignment requires comparison of MRV enabled
    # and MRV disabled on the Hard puzzle.
    #
    # If puzzles.txt contains the Hard puzzle as the first
    # puzzle, it will be used here.
    # ---------------------------------------------------------

    print("\nRunning ablation study on Puzzle 1...")

    run_ablation(puzzles[0])


if __name__ == "__main__":
    main()

INTELLIGENT SUDOKU SOLVER
CSP + AC-3 + Backtracking + MRV

Error: puzzles.txt was not found.
